In [3]:
from datetime import datetime, timedelta
import random
import sqlite3


def initialize_database(db_path: str = "smart_city.db") -> sqlite3.Connection:
  conn = sqlite3.connect(db_path)
  cursor = conn.cursor()

  cursor.execute("PRAGMA foreign_keys = ON;")

  cursor.executescript("""
        DROP TABLE IF EXISTS sensor_compliance;
        DROP TABLE IF EXISTS sensor_maintenance;
        DROP TABLE IF EXISTS readings;
        DROP TABLE IF EXISTS sensors;
        DROP TABLE IF EXISTS districts;
    """)

  cursor.executescript("""
        CREATE TABLE districts (
            district_id INTEGER PRIMARY KEY AUTOINCREMENT,
            district_name TEXT NOT NULL,
            city_zone TEXT NOT NULL,
            urban_density_index REAL NOT NULL
        );

        CREATE TABLE sensors (
            sensor_id INTEGER PRIMARY KEY AUTOINCREMENT,
            district_id INTEGER NOT NULL,
            sensor_model TEXT NOT NULL,
            installation_cost REAL NOT NULL CHECK(installation_cost >= 0.0),
            FOREIGN KEY (district_id) REFERENCES districts(district_id) ON DELETE CASCADE
        );

        CREATE TABLE readings (
            reading_id INTEGER PRIMARY KEY AUTOINCREMENT,
            sensor_id INTEGER NOT NULL,
            timestamp TEXT NOT NULL,
            alert_status TEXT NOT NULL CHECK(alert_status IN ('Normal', 'Warning', 'Critical')),
            weather_severity TEXT NOT NULL CHECK(weather_severity IN ('Clear', 'Moderate Haze', 'Heavy Smog', 'Hazardous')),
            temp_anomaly REAL NOT NULL,
            pm25_concentration REAL NOT NULL CHECK(pm25_concentration >= 0.0),
            wind_speed REAL CHECK(wind_speed >= 0.0),
            battery_voltage REAL CHECK(battery_voltage >= 0.0),
            FOREIGN KEY (sensor_id) REFERENCES sensors(sensor_id) ON DELETE CASCADE
        );

        CREATE TABLE sensor_maintenance (
            sensor_id INTEGER NOT NULL,
            maintenance_date TEXT NOT NULL,
            technician_id INTEGER NOT NULL,
            downtime_hours REAL NOT NULL CHECK(downtime_hours >= 0.0),
            PRIMARY KEY (sensor_id, maintenance_date),
            FOREIGN KEY (sensor_id) REFERENCES sensors(sensor_id) ON DELETE CASCADE
        );

        CREATE TABLE sensor_compliance (
            sensor_id INTEGER NOT NULL,
            audit_quarter TEXT NOT NULL,
            inspector_badge TEXT NOT NULL,
            compliance_score INTEGER NOT NULL CHECK(compliance_score BETWEEN 0 AND 100),
            PRIMARY KEY (sensor_id, audit_quarter),
            FOREIGN KEY (sensor_id) REFERENCES sensors(sensor_id) ON DELETE CASCADE
        );
    """)

  conn.commit()
  print("Smart City database schema created successfully with multiple compound keys.")
  return conn


def seed_districts(cursor: sqlite3.Cursor, count: int = 12) -> None:
  zones = [
      "North Ward",
      "Downtown",
      "Industrial Sector",
      "West Suburbs",
      "East Harbour",
  ]
  district_data = []
  for i in range(count):
    name = f"District {chr(65+i)}"
    zone = random.choice(zones)
    density = round(random.uniform(-40.0, 45.0), 2)
    district_data.append((name, zone, density))

  cursor.executemany(
      "INSERT INTO districts (district_name, city_zone, urban_density_index)"
      " VALUES (?, ?, ?);",
      district_data,
  )


def seed_sensors(cursor: sqlite3.Cursor, count: int = 40) -> None:
  models = ["EcoSense-X1", "AirGuard-Pro", "UrbanNode-Alpha", "MetroMetrics-v2"]
  sensor_data = []
  for i in range(count):
    d_id = random.randint(1, 12)
    model = random.choice(models)
    cost = round(random.uniform(450.0, 2500.0), 2)
    sensor_data.append((d_id, model, cost))

  sensor_data.append((3, "AirGuard-Pro", 1250.00))
  sensor_data.append((7, "EcoSense-X1", 890.50))

  cursor.executemany(
      "INSERT INTO sensors (district_id, sensor_model, installation_cost)"
      " VALUES (?, ?, ?);",
      sensor_data,
  )


def seed_readings(cursor: sqlite3.Cursor, count: int = 1400) -> None:
  alerts = ["Normal", "Warning", "Critical"]
  severities = ["Clear", "Moderate Haze", "Heavy Smog", "Hazardous"]
  start_date = datetime(2026, 1, 1)

  reading_data = []
  for _ in range(count):
    s_id = random.randint(1, 42)
    days_offset = random.randint(0, 365)
    t_stamp = (
        start_date
        + timedelta(
            days=days_offset,
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
        )
    ).strftime("%Y-%m-%d %H:%M:%S")

    alert = random.choices(alerts, weights=[75, 20, 5], k=1)[0]
    severity = random.choices(severities, weights=[50, 30, 15, 5], k=1)[0]
    temp_anom = round(random.uniform(-12.5, 18.2), 2)
    pm25 = round(random.uniform(2.1, 185.4), 2)

    wind = (
        None
        if random.random() < 0.05
        else round(random.uniform(0.0, 45.8), 2)
    )
    voltage = (
        None if random.random() < 0.03 else round(random.uniform(2.8, 4.2), 2)
    )

    reading_data.append(
        (s_id, t_stamp, alert, severity, temp_anom, pm25, wind, voltage)
    )

  cursor.executemany(
      """
        INSERT INTO readings (sensor_id, timestamp, alert_status, weather_severity, temp_anomaly, pm25_concentration, wind_speed, battery_voltage)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?);
    """,
      reading_data,
  )


def seed_maintenance(cursor: sqlite3.Cursor, target_count: int = 300) -> None:
  start_date = datetime(2026, 1, 1)
  maintenance_data = set()

  while len(maintenance_data) < target_count:
    s_id = random.randint(1, 42)
    days_offset = random.randint(1, 350)
    m_date = (start_date + timedelta(days=days_offset)).strftime("%Y-%m-%d")
    tech_id = random.randint(101, 115)
    downtime = round(random.uniform(0.5, 6.0), 2)
    maintenance_data.add((s_id, m_date, tech_id, downtime))

  cursor.executemany(
      """
        INSERT OR IGNORE INTO sensor_maintenance (sensor_id, maintenance_date, technician_id, downtime_hours)
        VALUES (?, ?, ?, ?);
    """,
      list(maintenance_data),
  )


def seed_compliance(cursor: sqlite3.Cursor, target_count: int = 150) -> None:
  quarters = ["2026-Q1", "2026-Q2", "2026-Q3", "2026-Q4"]
  compliance_data = set()

  while len(compliance_data) < target_count:
    s_id = random.randint(1, 42)
    qtr = random.choice(quarters)
    inspector = f"Insp-{random.randint(1000, 1050)}"
    score = random.randint(65, 100)
    compliance_data.add((s_id, qtr, inspector, score))

  cursor.executemany(
      """
        INSERT OR IGNORE INTO sensor_compliance (sensor_id, audit_quarter, inspector_badge, compliance_score)
        VALUES (?, ?, ?, ?);
    """,
      list(compliance_data),
  )


def verify_database(cursor: sqlite3.Cursor) -> None:
  cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
  tables = cursor.fetchall()
  print("\n--- Database Verification Report ---")
  for table in tables:
    t_name = table[0]
    cursor.execute(f"SELECT COUNT(*) FROM {t_name};")
    count = cursor.fetchone()[0]
    print(f"Table '{t_name}': {count} rows successfully generated.")


if __name__ == "__main__":
  random.seed(42)

  connection = initialize_database("smart_city.db")
  db_cursor = connection.cursor()

  seed_districts(db_cursor, count=12)
  seed_sensors(db_cursor, count=40)
  seed_readings(db_cursor, count=1400)
  seed_maintenance(db_cursor, target_count=300)
  seed_compliance(db_cursor, target_count=150)

  connection.commit()
  verify_database(db_cursor)
  connection.close()

Smart City database schema created successfully with multiple compound keys.

--- Database Verification Report ---
Table 'sqlite_sequence': 3 rows successfully generated.
Table 'districts': 12 rows successfully generated.
Table 'sensors': 42 rows successfully generated.
Table 'readings': 1400 rows successfully generated.
Table 'sensor_maintenance': 300 rows successfully generated.
Table 'sensor_compliance': 101 rows successfully generated.
